# Import Library and Dataset

In [32]:
# Import essential library
import pandas as pd
import numpy as np
import re

In [33]:
# Import datasets
general_df = pd.read_csv("goemotions_mrm8488_train.csv")
domain_df = pd.read_csv("stockemotions_full.csv", skiprows = 1)

print(general_df.shape)
print(domain_df.shape)
print(general_df.head(10))
print(domain_df.head(10))

(211225, 37)
(10000, 7)
                                                text       id  \
0                                    That game hurt.  eew5j0j   
1   >sexuality shouldn’t be a grouping category I...  eemcysk   
2     You do right, if you don't care then fuck 'em!  ed2mah1   
3                                 Man I love reddit.  eeibobj   
4  [NAME] was nowhere near them, he was by the Fa...  eda6yn6   
5  Right? Considering it’s such an important docu...  eespn2i   
6  He isn't as big, but he's still quite popular....  eczuekb   
7  That's crazy; I went to a super [RELIGION] hig...  ed5tx8y   
8                                that's adorable asf  ef961hv   
9  "Sponge Blurb Pubs Quaw Haha GURR ha AAa!" fin...  edl7cr3   

                author             subreddit    link_id   parent_id  \
0                Brdd9                   nrl  t3_ajis4z  t1_eew18eq   
1          TheGreen888      unpopularopinion  t3_ai4q37   t3_ai4q37   
2             Labalool           confessions  t

# Basic Statistics


In [34]:
# The shape of the datasets
print(f'General Dataset: {general_df.shape}')
print(f'Domain Dataset: {domain_df.shape}')

General Dataset: (211225, 37)
Domain Dataset: (10000, 7)


In [35]:
# Check datatype
print(general_df.info())
print("________________________________\n")
print(domain_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211225 entries, 0 to 211224
Data columns (total 37 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   text                  211225 non-null  object 
 1   id                    211225 non-null  object 
 2   author                211225 non-null  object 
 3   subreddit             211225 non-null  object 
 4   link_id               211225 non-null  object 
 5   parent_id             211225 non-null  object 
 6   created_utc           211225 non-null  float64
 7   rater_id              211225 non-null  int64  
 8   example_very_unclear  211225 non-null  bool   
 9   admiration            211225 non-null  int64  
 10  amusement             211225 non-null  int64  
 11  anger                 211225 non-null  int64  
 12  annoyance             211225 non-null  int64  
 13  approval              211225 non-null  int64  
 14  caring                211225 non-null  int64  
 15  

In [36]:
# Checking missing value
print(f'General Dataset: {general_df.isnull().sum()}')
print("________________________________\n")
print(f'Domain Dataset: {domain_df.isnull().sum()}')

General Dataset: text                    0
id                      0
author                  0
subreddit               0
link_id                 0
parent_id               0
created_utc             0
rater_id                0
example_very_unclear    0
admiration              0
amusement               0
anger                   0
annoyance               0
approval                0
caring                  0
confusion               0
curiosity               0
desire                  0
disappointment          0
disapproval             0
disgust                 0
embarrassment           0
excitement              0
fear                    0
gratitude               0
grief                   0
joy                     0
love                    0
nervousness             0
optimism                0
pride                   0
realization             0
relief                  0
remorse                 0
sadness                 0
surprise                0
neutral                 0
dtype: int64
________

# Clean Dataset


In [37]:

# Fiter ()"example_very_unclear" = True) as it do not have any label
general_df = general_df[~general_df["example_very_unclear"]]
print(general_df["example_very_unclear"].head(10))

0     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
Name: example_very_unclear, dtype: bool


In [38]:
# Applying rarest-label strategy to balance the dataset
min_vote = 2
emotion_cols = general_df.columns[9:].tolist()
agg = general_df.groupby('text')[emotion_cols].sum()
mask = agg >= min_vote
agg = agg[mask.any(axis = 1)]
mask = mask[mask.any(axis = 1)]
rare_rank = mask.sum().rank(method = "first")
scores = (mask * rare_rank).where(mask)
agg["label"] = scores.idxmin(axis=1)
general_clean = agg.reset_index()[["text", "label"]]
print(general_clean.head(10))
print(general_clean.shape)


                                                text        label
0   "If you don't wear BROWN AND ORANGE...YOU DON...    annoyance
1   "What do Scottish people look like?" How I wo...         love
2     ### A surprise, to be sure, but a welcome one      surprise
3   '*Pray*, v. To ask that the laws of the unive...      neutral
4   >it'll get invaded by tankie, unfortunately. ...      neutral
5   And not all children's hospitals need the sam...     approval
6                Best number! [NAME], [NAME], [NAME]   admiration
7   Calm down and relax are the worst things to s...    annoyance
8   Change is hard. Find comfort in victory, even...      neutral
9   Don't be so stupid. Terrorism is inherently p...  disapproval
(53994, 2)


In [39]:
duplicate_value = general_clean[general_clean.duplicated("text", keep=False)].sort_values("text")
print("Duplicate Rows:", len(duplicate_value), "| Duplicate Text", duplicate_value["text"].nunique())

Duplicate Rows: 0 | Duplicate Text 0


## Domain Dataset - StockEmotions

In [40]:
#Check duplicate value
duplicate_data = domain_df[domain_df.duplicated()]
print(duplicate_value)

Empty DataFrame
Columns: [text, label]
Index: []


In [41]:
domain_df["text"] = (domain_df["processed"].astype(str).str.replace(r"[\u200d\ufe0f]", "", regex=True).str.replace(r"[\U0001F000-\U0001FAFF\U0001F1E6-\U0001F1FF\u2190-\u21FF\u2600-\u27BF\u2B00-\u2BFF]", " ", regex=True).str.replace(r"\$(?=[A-Za-z])", "", regex=True).str.replace(r"\s+", " ", regex=True).str.strip())
domain_df.to_csv("domain_df.csv", index = True, encoding = "utf-8")